# AI-Assisted Transition State Search

In this notebook, we compare classical and AI-based approaches for finding transition states (TS), with a focus on:

- Standard non-AI methods
- TS prediction (partially) using AI
- Ranking and refinement strategies with and without AI
- Practical workflows

We will:
1. Establish a classical baseline (NEB)
2. Exchange the electronic structure backend with an MLIP
3. Generate TS candidates using only AI (tsdiff)
4. Discuss refinement strategies and where AI can help

In [1]:
import torch
import torchani
#import numpy as np

from ase import Atoms
from ase.mep import NEB, idpp_interpolate
#from ase.neb import NEB, idpp_interpolate
from ase.optimize import BFGS
from ase.calculators.calculator import Calculator, all_changes
#from ase.calculators.emt import EMT
from xtb.ase.calculator import XTB
#from ase.build import molecule

/home/marco/miniconda3/envs/tsdiff_intro/lib/python3.13/site-packages/torchani/csrc/__init__.py:56: UserWarning: The extensions: ['cuaev', 'mnp', 'cell_list'] are not installed and will not be available. To install the extensions first install the CUDA Toolkit, and afterwards  run `ani build-extensions` To suppress warn set the env var TORCHANI_NO_WARN_EXTENSIONS=1 For example, if using bash, you may add `export TORCHANI_NO_WARN_EXTENSIONS=1` to your .bashrc
  warnings.warn(


## Transition State Search: Practical Perspective

Classical methods:
- NEB: robust but expensive and initialization-sensitive

AI-based methods:
- Direct TS prediction: fast, but ranking is often imperfect

Key idea:
> The correct TS is often in the candidate set, but not ranked first.

We focus on:
- Candidate generation
- Ranking
- Efficient refinement

In [24]:
# Structure definitions for ase for a proton transfer between two water molecules
reactant = Atoms(
    symbols=["O","H","H","O","H","H"],
    positions=[
        [0.000,0.000,0.000],
        [0.757,0.586,0.000],
        [-0.757,0.586,0.000],
        [2.500,0.000,0.000],
        [3.257,0.586,0.000],
        [1.743,0.586,0.000]
    ]
)

product = Atoms(
    symbols=["O","H","H","O","H","H"],
    positions=[
        [0.000,0.000,0.000],
        [0.757,0.586,0.000],
        [1.200,0.000,0.000],
        [2.500,0.000,0.000],
        [3.257,0.586,0.000],
        [1.743,0.586,0.000]
    ]
)

In [21]:
# this is not a good example, because the xtb calculator doesn't converge,
# while the torchani calculator converges in 4 iterations
reactant2 = Atoms(
    symbols=["O","O","H","H","H","H","H"],
    positions=[
        [-1.200,  0.000,  0.000],  # O1 (hydronium)
        [ 1.200,  0.000,  0.000],  # O2 (water)

        [-1.600,  0.900,  0.000],  # H3
        [-1.600, -0.900,  0.000],  # H4

        [-0.300,  0.000,  0.000],  # H5 transferring proton

        [ 1.600,  0.900,  0.000],  # H6
        [ 1.600, -0.900,  0.000],  # H7
    ]
)

product2 = Atoms(
    symbols=["O","O","H","H","H","H","H"],
    positions=[
        [-1.200,  0.000,  0.000],  # O1 (water)
        [ 1.200,  0.000,  0.000],  # O2 (hydronium)

        [-1.600,  0.900,  0.000],  # H3
        [-1.600, -0.900,  0.000],  # H4

        [ 0.300,  0.000,  0.000],  # H5 transferred proton

        [ 1.600,  0.900,  0.000],  # H6
        [ 1.600, -0.900,  0.000],  # H7
    ]
)

In [4]:
# raw form of standard organic SN2 reaction, which might be a good example later
"""
# SN2: Cl- + CH3Br → CH3Cl + Br-

from ase.build import molecule

ch3br = molecule("CH3Br")
cl = Atoms("Cl", positions=[[3.0, 0.0, 0.0]])

reactant = ch3br + cl

# Product: Cl replaces Br
product = reactant.copy()
product.positions[-1] = product.positions[0] + np.array([-1.8, 0, 0])

# Use xTB (already uses effective core approximations internally)

attach_calc(reactant)
attach_calc(product)

# Then same NEB workflow as above
"""

'\n# SN2: Cl- + CH3Br → CH3Cl + Br-\n\nfrom ase.build import molecule\n\nch3br = molecule("CH3Br")\ncl = Atoms("Cl", positions=[[3.0, 0.0, 0.0]])\n\nreactant = ch3br + cl\n\n# Product: Cl replaces Br\nproduct = reactant.copy()\nproduct.positions[-1] = product.positions[0] + np.array([-1.8, 0, 0])\n\n# Use xTB (already uses effective core approximations internally)\n\nattach_calc(reactant)\nattach_calc(product)\n\n# Then same NEB workflow as above\n'

## Baseline: Nudged Elastic Band (NEB)

We compute a reaction path between reactant and product.

In [25]:
# It took me a long time to find something, which can use neb with an electronic
# structure backend as well as an MLIP, and setting it up was even worse!

# Anyway this runs neb with xtb
def attach_xtb(atoms):
    #atoms.calc = EMT()
    atoms.calc = XTB(method="GFN2-xTB")
    return atoms

def run_neb(reactant, product, method, n_images=5):
    images = [reactant.copy()]

    # this didn't work, because this kind of simple interpolation
    # yielded guesses, which were so bad that neb didn't converge
    # even with small step sizes, etc.
    #for i in range(n_images):
    #    img = reactant.copy()
    #    img.positions = reactant.positions + (i+1)/(n_images+1)*(product.positions - reactant.positions)
    #    images.append(img)

    for i in range(n_images):
        images.append(reactant.copy())

    images.append(product.copy())

    # attach calculator ("attach" methods)
    for img in images:
        method(img)

    # For some reason different calculators are recommended, but
    # for the current setup this works as well
    neb = NEB(images, allow_shared_calculator=True)

    # This does some proper interpolation for the neb images
    idpp_interpolate(images)

    # choose a rather small step size here, because it won't converge otherwise
    opt = BFGS(neb, maxstep=0.02)

    # This will literally run forever, if you don't give a max_iter
    opt.run(fmax=0.1, steps=500)

    # assume middle image to be TS for now
    return images[len(images)//2], images

ts_xtb, neb_images_xtb = run_neb(reactant, product, attach_xtb)
# TODO: output and visualize structure

      Step     Time          Energy          fmax
BFGS:    0 13:26:58     -271.190894        5.501546
BFGS:    1 13:26:58     -271.468028        5.401459
BFGS:    2 13:26:58     -271.732381        5.262142
BFGS:    3 13:26:58     -271.987200        5.047736
BFGS:    4 13:26:59     -272.232155        4.720672
BFGS:    5 13:26:59     -272.465971        4.232987
BFGS:    6 13:26:59     -272.685469        3.526028
BFGS:    7 13:26:59     -272.880933        2.759582
BFGS:    8 13:26:59     -273.022676        2.606324
BFGS:    9 13:26:59     -273.124152        2.429523
BFGS:   10 13:26:59     -273.209067        2.168336
BFGS:   11 13:26:59     -273.288113        2.064542
BFGS:   12 13:26:59     -273.365759        2.008390
BFGS:   13 13:26:59     -273.442807        1.966512
BFGS:   14 13:26:59     -273.518823        1.933089
BFGS:   15 13:27:00     -273.593343        1.904927
BFGS:   16 13:27:00     -273.666129        1.880115
BFGS:   17 13:27:00     -273.737431        1.858963
BFGS:   18 13:

In [26]:
# And the following two cells run neb with an MLIP

# setup MLIP model
# if your device has a GPU I highly recommend to use it, by setting the device here
# and previously importing the appropriate pytorch expansion into your conda environment.
# For an nvidia GPU this would be torch.device("cuda")
device = torch.device("cpu")
ani_model = torchani.models.ANI2x().to(device)
species_converter = ani_model.species_converter

# mocked calculator class for MLIP
class ANICalculator(Calculator):
    implemented_properties = ['energy', 'forces']

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)

        coords = torch.tensor(
            atoms.get_positions(),
            dtype=torch.float32,
            device=device,
            requires_grad=True
        ).unsqueeze(0)

        atomic_numbers = torch.tensor(
            [atoms.get_atomic_numbers()],
            dtype=torch.long,
            device=device
        )

        energy = ani_model((atomic_numbers, coords)).energies
        forces = -torch.autograd.grad(energy.sum(), coords)[0]

        self.results['energy'] = energy.item()
        self.results['forces'] = forces.squeeze(0).cpu().detach().numpy()


def attach_ani(atoms):
    atoms.calc = ANICalculator()
    return atoms

In [27]:
# run neb with MLIP
ts_ani, neb_images_ani = run_neb(reactant, product, attach_ani)
# TODO: output and visualize structure

      Step     Time          Energy          fmax
BFGS:    0 13:27:11     -152.553619        0.267734
BFGS:    1 13:27:11     -152.556641        0.266925
BFGS:    2 13:27:11     -152.572311        0.259177
BFGS:    3 13:27:12     -152.587418        0.248928
BFGS:    4 13:27:12     -152.601318        0.234359
BFGS:    5 13:27:12     -152.613739        0.217824
BFGS:    6 13:27:12     -152.624496        0.200151
BFGS:    7 13:27:12     -152.633133        0.170634
BFGS:    8 13:27:12     -152.639130        0.144712
BFGS:    9 13:27:12     -152.642456        0.159791
BFGS:   10 13:27:12     -152.642563        0.187341
BFGS:   11 13:27:12     -152.640518        0.196490
BFGS:   12 13:27:13     -152.634308        0.279180
BFGS:   13 13:27:13     -152.627975        0.354042
BFGS:   14 13:27:13     -152.624573        0.390008
BFGS:   15 13:27:13     -152.624420        0.390082
BFGS:   16 13:27:13     -152.625916        0.367939
BFGS:   17 13:27:13     -152.627380        0.340704
BFGS:   18 13:

## Second example

In [22]:
ts_xtb, neb_images_xtb = run_neb(reactant2, product2, attach_xtb)

      Step     Time          Energy          fmax
BFGS:    0 13:25:23     -282.268562        8.192666
BFGS:    1 13:25:24     -282.538464        6.067588
BFGS:    2 13:25:24     -282.781677        5.556876
BFGS:    3 13:25:24     -282.761788        4.928815
BFGS:    4 13:25:24     -282.625180        3.751306
BFGS:    5 13:25:24     -282.506473        3.244142
BFGS:    6 13:25:24     -282.393496        2.870844
BFGS:    7 13:25:24     -282.230080        2.512393
BFGS:    8 13:25:24     -282.050215        2.583285
BFGS:    9 13:25:24     -281.873367        3.040629
BFGS:   10 13:25:24     -281.749842        4.444768
BFGS:   11 13:25:24     -281.661025       10.173360
BFGS:   12 13:25:24     -281.709694        9.769197
BFGS:   13 13:25:25     -281.744051       11.389354
BFGS:   14 13:25:25     -281.765003       10.267148
BFGS:   15 13:25:25     -281.770761       10.069704
BFGS:   16 13:25:25     -281.770337        9.571355
BFGS:   17 13:25:25     -281.764353        7.844952
BFGS:   18 13:

In [23]:
ts_ani, neb_images_ani = run_neb(reactant2, product2, attach_ani)

      Step     Time          Energy          fmax
BFGS:    0 13:25:56     -153.292816        0.150666
BFGS:    1 13:25:56     -153.293243        0.146795
BFGS:    2 13:25:57     -153.296524        0.108356
BFGS:    3 13:25:57     -153.298660        0.093048


As can be seen the performances are very different, stressing the importance of the electronic structure backend and especially the initial guess. This is a big downside, so models providing a proper guess structure are very important even for seemingly simple cases!

### Refinement of the guessed structures

Standard: Use TS guess and optimize purely with electronic structure backend

In [16]:
def refine_xtb(ts):
    ts = ts.copy()
    attach_xtb(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.02)
    return ts

#ts_refined_std = refine_xtb(ranked[0][0])

Surrogate accelaration by preoptimizing with MLIP

In [17]:
def refine_ml_then_xtb(ts):
    ts = ts.copy()
    
    # Step 1: ANI refinement
    attach_ani(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.02)
    
    # Step 2: xTB refinement
    attach_xtb(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.02)
    
    return ts

#ts_refined_ml = refine_ml_then_xtb(ranked[0][0])

Accelaration of initial Hessian guess (using MLIP)

In [18]:
# For this I found that Hessians are typically build from MLIP information

## Ranking TS Candidates

We rank candidates based on approximate energies or heuristics.

Observation:
- Top-1 is sometimes wrong
- Correct TS may appear in top-2 or top-3

Hence, when you predict a transition state, you should always generate multiple candidates.